# 05 · A/B/C 비교 측정

문항 수와 실행 설정은 **`config/experiment.json`**에서 바꾼다. 노트북 코드는 수정하지 않는다.
현재 기본 설정은 개발 NQ·HotpotQA **각 20문항(총 40)**, 평가 **각 50문항(총 100)**이다.
`split`이 `dev`면 개발 설정, `eval`이면 평가 설정을 사용한다. 각 방법은 같은 질문을 받는다.

1. 설정 파일의 문항 수와 `split`을 지정한다.
2. **decision_LLM (Python 3.14, CUDA)** 커널에서 이 노트북을 처음부터 끝까지 실행한다.
3. `06_compare_results.ipynb`를 실행하면 최신 완료 시스템 런의 비교 그림이 나온다.

A는 RetroMAE 검색, B는 같은 후보를 DeBERTa로 재정렬, C는 Jev 자연어 트리 탐색이다.
공통 원문·질문·평가기·GPT-6 Luna를 사용한다. 품질 캐시 평가, 새 검색 시간, 새 검색 + LLM 시간은 별도로 기록한다.
설정의 실행 스위치가 `true`면 Jev 및 GPT 유료 API를 호출한다. LLM 최대 호출 수는 문항 수 × 3방법 × 반복 수로 자동 계산한다.
C의 초기 정책은 아직 보정 전이며 평가 전에 고정해야 한다.

`save_raw_logs: true`는 실제 로컬 모델 입력/출력과 API 본문을 `results/raw/`에 저장한다. 단계·질문·방법·반복별로 연결되며 실패도 기록한다. 입력/출력 기록 시간은 관측 지연에 포함되고 별도 기록되므로 기존 기록 없는 파일럿과 시간 조건을 구분한다. 로그 구조는 [원시 로그 안내](../research/raw-logs.md)를 참고한다.

In [ ]:
from pathlib import Path
from uuid import uuid4
import sys, os, json, time
from dotenv import load_dotenv
ROOT = Path.cwd()
if not (ROOT/'config/c.json').exists(): ROOT = ROOT.parent
assert (ROOT/'config/c.json').exists()
load_dotenv(ROOT/'.env', override=False, encoding='utf-8-sig')
sys.path.insert(0, str(ROOT))
# Windows Jupyter: load Pandas/PyArrow before Torch to avoid arrow.dll crashes.
import pandas as pd
# Refresh definitions cached by an already-running notebook kernel.
# Reload dependencies first, then bind the current classes/functions below.
import importlib
for module_name in ('rag_baselines','rag_baselines.data','rag_baselines.store',
                    'rag_baselines.generation','rag_baselines.jev','rag_baselines.c_tree',
                    'rag_baselines.raw_logging','rag_baselines.models',
                    'rag_baselines.c_retrieval','rag_baselines.evaluation'):
    importlib.reload(importlib.import_module(module_name))
import httpx
from rag_baselines.raw_logging import RawLog
from rag_baselines.data import read_json, read_jsonl, write_json, write_jsonl, file_sha256
from rag_baselines.models import Baselines
from rag_baselines.store import DocumentStore
from rag_baselines.c_tree import load_tree
from rag_baselines.c_retrieval import JevRetrieval, ComparativeRetrieval
from rag_baselines.jev import JevClient
from rag_baselines.evaluation import (evaluate_comparative_quality, measure_retrieval, measure_system,
                                     result_table, runtime_manifest, save_run, notebook_source_sha256)
from rag_baselines.generation import build_prompt
CONFIG = read_json(ROOT/'config/baselines.json')
C_CONFIG = read_json(ROOT/'config/c.json')
TAXONOMY = read_json(ROOT/'config/c_taxonomy.json')
GENERATION = read_json(ROOT/'config/generation.json')
pd.set_option('display.max_colwidth', 100)
print('Workspace:', ROOT)
print('Keys configured:', {'Jev':bool(os.environ.get('TYPESAFE_API_KEY')), 'OpenAI':bool(os.environ.get('OPENAI_API_KEY'))})

In [ ]:
EXPERIMENT_PATH = ROOT/'config/experiment.json'
EXPERIMENT_CONFIG = read_json(EXPERIMENT_PATH)
EXPERIMENT_SNAPSHOT = {'path':str(EXPERIMENT_PATH.relative_to(ROOT)),
                       'sha256':file_sha256(EXPERIMENT_PATH), 'parameters':EXPERIMENT_CONFIG}
SPLIT = EXPERIMENT_CONFIG['split']
if SPLIT not in ('dev','eval'):
    raise ValueError('experiment.json: split must be dev or eval.')
QUESTIONS_PER_TASK = EXPERIMENT_CONFIG['questions_per_task'][SPLIT]
REPEATS = EXPERIMENT_CONFIG['repeats']
CORPUS_SIZE = EXPERIMENT_CONFIG['corpus_size']
RETRIEVE_K = EXPERIMENT_CONFIG['retrieve_k']
EVIDENCE_K = EXPERIMENT_CONFIG['evidence_k']
RUN_RETRIEVAL = EXPERIMENT_CONFIG['run_retrieval']
RUN_GENERATION = EXPERIMENT_CONFIG['run_generation']
SAVE_RAW_LOGS = EXPERIMENT_CONFIG.get('save_raw_logs',True)
METHODS = ('A','B','C')
for name,value in {'questions_per_task':QUESTIONS_PER_TASK, 'repeats':REPEATS,
                   'corpus_size':CORPUS_SIZE, 'retrieve_k':RETRIEVE_K, 'evidence_k':EVIDENCE_K}.items():
    if type(value) is not int or value < 1:
        raise ValueError(f'experiment.json: {name} must be a positive integer.')
if any(type(v) is not bool for v in (RUN_RETRIEVAL,RUN_GENERATION,SAVE_RAW_LOGS)):
    raise ValueError('experiment.json: run switches must be true or false.')
if EVIDENCE_K > min(RETRIEVE_K,CORPUS_SIZE):
    raise ValueError('evidence_k must not exceed retrieve_k or corpus_size.')
if RUN_GENERATION and not RUN_RETRIEVAL:
    raise ValueError('System measurement needs run_retrieval=true.')
assert GENERATION['encoding_name'] == C_CONFIG['search']['encoding_name']
assert C_CONFIG['search']['evidence_text_tokens'] == 2000
manifest = read_json(ROOT/'data/prepared/manifest.json')
corpus_size = CORPUS_SIZE
if str(corpus_size) not in manifest['corpora']:
    raise ValueError('Choose a prepared corpus_size (2000, 10000, 40000).')
entry = manifest['corpora'][str(corpus_size)]
corpus_path = ROOT/entry['path']
assert file_sha256(corpus_path) == entry['sha256']
all_questions = read_jsonl(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')
questions = []
for task in CONFIG['qa']['tasks']:
    available = [q for q in all_questions if q['task']==task]
    if QUESTIONS_PER_TASK > len(available):
        raise ValueError(f'{SPLIT}/{task}: requested {QUESTIONS_PER_TASK}, available {len(available)}.')
    questions.extend(available[:QUESTIONS_PER_TASK])
labels = read_jsonl(ROOT/f'data/prepared/labels_{SPLIT}.jsonl')
MAX_LLM_API_CALLS = len(questions)*len(METHODS)*REPEATS
EXPERIMENT_SNAPSHOT['selected_question_ids'] = [q['id'] for q in questions]
display(pd.DataFrame(questions).groupby('task').size().rename('selected_questions').to_frame())
display(pd.DataFrame(questions).head(6))
print('Split:',SPLIT,'| total questions:',len(questions),'| repeats:',REPEATS)
print('Run switches:',{'retrieval':RUN_RETRIEVAL,'generation':RUN_GENERATION})
print('Maximum LLM API calls:',MAX_LLM_API_CALLS,'| Jev calls per query:',C_CONFIG['search']['max_api_calls'])
print('Pinned Jev:',C_CONFIG['model'],'| common generator:',GENERATION['model'])

## 사전 준비와 잘림 검증

04에서 완성한 트리를 읽는다. 이 노트북은 트리를 새로 적재하지 않는다. 임베딩 인덱스·Jev 적재·모델 로딩 비용을 온라인 시간과 분리한다. C의 자료 설명은 제목과 원문의 첫 24단어이며, Score 단계와 LLM에는 전체 원문 문단을 전달한다. 설명에 빠진 정보는 검색 누락을 일으킬 수 있다.

In [ ]:
engine = None
jev_client = None
jev_http = None
audit = None                         # Token truncation audit, separate from RAW_LOG.
RAW_LOG = None
RAW_SESSION_META = {'enabled':SAVE_RAW_LOGS,'session':None,
                    'timing':'diagnostic input/output capture is included in measured wall time'}
if RUN_RETRIEVAL:
    store = DocumentStore(corpus_path)
    tree, preparation = load_tree(ROOT,store,TAXONOMY,C_CONFIG)
    if SAVE_RAW_LOGS:
        RAW_LOG = RawLog(ROOT,manifest={'notebook':'05_measure_abc.ipynb','experiment':EXPERIMENT_SNAPSHOT,
            'corpus_sha256':entry['sha256'],'generation':GENERATION,'c_config':C_CONFIG},
            prices={'openai':GENERATION['prices'],'jev':C_CONFIG['prices']})
        RAW_SESSION_META['session'] = str(RAW_LOG.directory.relative_to(ROOT))
        # Labels are evaluator-only snapshots, never passed to either retriever or generator.
        for path in [EXPERIMENT_PATH,ROOT/'config/baselines.json',ROOT/'config/c.json',
                     ROOT/'config/c_taxonomy.json',ROOT/'config/generation.json',corpus_path,
                     ROOT/'data/prepared/manifest.json',ROOT/'data/prepared/preparation_metrics.json',
                     ROOT/f'data/prepared/questions_{SPLIT}.jsonl',ROOT/f'data/prepared/labels_{SPLIT}.jsonl',
                     ROOT/'notebook/05_measure_abc.ipynb',*sorted((ROOT/'rag_baselines').glob('*.py'))]:
            RAW_LOG.snapshot(path)
        with RAW_LOG.scope(phase='preparation',method='C'):
            RAW_LOG.event('tree','reused',{'preparation':preparation,'actual_new_api_calls':0})
            for name in ('tree.json','manifest.json','checkpoint.json','checkpoint_hash.json'):
                RAW_LOG.snapshot(Path(preparation['directory'])/name)
        print('Raw logs:',RAW_LOG.directory)
    baseline = Baselines(ROOT,retrieve_k=RETRIEVE_K,evidence_k=EVIDENCE_K,raw_log=RAW_LOG)
    baseline.build_index(corpus_path)
    audit = baseline.truncation_audit(read_jsonl(ROOT/'data/prepared/questions_dev.jsonl') +
                                      read_jsonl(ROOT/'data/prepared/questions_eval.jsonl'))
    if RAW_LOG:
        RAW_LOG.result('truncation_audit',audit)
        with RAW_LOG.scope(phase='warmup',method='AB'):
            baseline.warmup(questions[0]['question'])
    else:
        baseline.warmup(questions[0]['question'])
    if RAW_LOG:
        jev_http = httpx.Client(timeout=C_CONFIG['timeout_seconds'],event_hooks=RAW_LOG.http_hooks)
    jev_client = JevClient(C_CONFIG,http_client=jev_http)
    decision = JevRetrieval(ROOT,baseline.store,tree,preparation,C_CONFIG,client=jev_client)
    engine = ComparativeRetrieval(baseline,decision)
    if RAW_LOG:
        RAW_LOG.result('runtime',runtime_manifest(ROOT,engine))
    display(pd.DataFrame(audit).T)
    display(pd.DataFrame([{'method':'A/B','preparation_seconds':baseline.index_info['build_seconds'],
                          'api_cost_usd':None,'input_tokens':baseline.index_info['document_tokens']['nonpad'],
                          'cache_hit':baseline.index_info['cache_hit']},
                         {'method':'C','preparation_seconds':preparation['prepare_seconds'],
                          'api_cost_usd':preparation['api_cost_usd'],'input_tokens':preparation['api_usage']['input_tokens'],
                          'cache_hit':preparation['cache_hit']}]))
    print('C taxonomy authoring time:',preparation['human_taxonomy_authoring_seconds'],'(unmeasured)')
    print('C guards: no input truncation; UTF-8 byte limits are not exact Jev token counts.')
else:
    print('검색 미실행. 키·트리 없이 자료와 설정만 확인했습니다.')

## 저장된 검색 결과로 품질 평가

A/B는 dense 후보 목록이 동일한지 검증한다. C는 별도의 판단 캐시를 사용한다. 캐시 최초 생성에는 Jev 요금이 발생하며, 재사용에는 실제 호출이 없다. 이 실행의 검색 시간은 지연 평가에 사용하지 않는다. 정답 라벨은 검색이 끝난 뒤 평가에만 사용한다.

In [ ]:
quality = []
quality_directory = None
if engine:
    quality = evaluate_comparative_quality(engine,questions,labels,audit=RAW_LOG)
    for q in questions:
        pair = [r for r in quality if r['question_id']==q['id'] and r['method'] in ('A','B')]
        assert [(p['id'],p['dense_score']) for p in pair[0]['candidates']] == [(p['id'],p['dense_score']) for p in pair[1]['candidates']]
    for r in quality:
        engine.store.validate_evidence(r)
    quality_table = result_table(quality)
    display(quality_table[['task','method','complete_evidence_hit','complete_candidate_hit',
                           'best_group_coverage','evidence_count','retrieval_api_calls','api_cost_usd']])
    meta = runtime_manifest(ROOT,engine)
    meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'cached_quality_abc','split':SPLIT,'truncation_audit':audit,
                 'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
    quality_directory = save_run(ROOT,quality,meta,'05_measure_abc.ipynb')
    print('Quality saved:',quality_directory)

## 실제 검색 시간·입력량·비용

A/B/C가 각 질문을 독립적으로 새로 검색한다. 방법 순서를 무작위로 섞는다. A/B는 로컬 GPU, C는 외부 API이므로 C 시간에는 네트워크도 포함된다. 이 결과는 기록한 배포 환경의 비교다. 로컬 GPU 금전 비용은 미측정이며 API 비용과 별도로 해석한다. 완료한 질문별 원시 결과를 즉시 저장한다.

In [ ]:
online = []
online_directory = None
if engine:
    partial_directory = ROOT/'results/partial'/uuid4().hex
    partial_directory.mkdir(parents=True,exist_ok=False)
    meta = runtime_manifest(ROOT,engine)
    meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'online_retrieval_abc','split':SPLIT,'repeats':REPEATS,'truncation_audit':audit,
                 'notebook_source_sha256':notebook_source_sha256(ROOT/'notebook/05_measure_abc.ipynb'),
                 'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
    write_json(partial_directory/'manifest.json',meta)
    def save_retrieval_record(record):
        with (partial_directory/'retrieval.jsonl').open('a',encoding='utf-8') as stream:
            stream.write(json.dumps(record,ensure_ascii=False)+'\n')
    online = measure_retrieval(engine,questions,labels,repeats=REPEATS,seed=CONFIG['seed'],
                              methods=METHODS,on_result=save_retrieval_record,audit=RAW_LOG)
    online_directory = save_run(ROOT,online,meta,'05_measure_abc.ipynb')
    write_json(partial_directory/'completed.json',{'run_directory':str(online_directory)})
    online_table = result_table(online)
    display(online_table[['task','method','retrieval_total_ms','complete_evidence_hit','evidence_count',
                          'jev_input_tokens','retrieval_api_calls','api_cost_usd','c_stop_reason']])
    print('Online saved:',online_directory)

In [ ]:
if online:
    throughput = []
    for method,group in online_table.groupby('method'):
        row = {'method':method,'queries_per_wall_second':len(group)/(group.retrieval_total_ms.sum()/1000)}
        if method in ('A','B'):
            row['query_tokens_per_stage_second'] = group.retriever_input_tokens.sum()/(group.query_encode_ms.sum()/1000)
        if method=='B':
            row['rerank_pairs_per_stage_second'] = len(group)*min(RETRIEVE_K,corpus_size)/(group.rerank_ms.sum()/1000)
            row['rerank_tokens_per_stage_second'] = group.reranker_input_tokens.sum()/(group.rerank_ms.sum()/1000)
        if method=='C':
            request_seconds = sum(sum(c['seconds'] for c in r['trace']['calls']) for r in online if r['method']=='C')
            row['jev_input_tokens_per_request_second'] = group.jev_input_tokens.sum()/request_seconds
            row['jev_requests_per_request_second'] = group.retrieval_api_calls.sum()/request_seconds
        throughput.append(row)
    throughput_table = pd.DataFrame(throughput)
    throughput_table.to_csv(online_directory/'model_throughput.csv',index=False)
    display(throughput_table)
    prompt_previews = [{**build_prompt(r,encoding_name=GENERATION['encoding_name']),
                       'question_id':r['question_id'],'method':r['method']} for r in online]
    assert all(p['evidence_text_tokens']<=2000 for p in prompt_previews)
    write_jsonl(online_directory/'prompt_previews.jsonl',prompt_previews)
    write_json(online_directory/'truncation_audit.json',audit)
    display(pd.DataFrame([{'question_id':p['question_id'],'method':p['method'],
                          'evidence_text_tokens':p['evidence_text_tokens'],'passage_ids':p['passage_ids']} for p in prompt_previews]))
    c_example = next(r for r in online if r['method']=='C')
    display(pd.DataFrame([{'id':p['id'],'title':p['title'],'original_text':p['content']} for p in c_example['evidence']]))

## 공통 GPT-6 Luna 전체 시스템 측정

새 검색 → 같은 프롬프트 → 같은 LLM API까지 실제로 측정한다. 정답 EM/F1, 실제 청구 토큰,
Jev/LLM/합산 API 비용을 분리해 저장한다. 근거 예산은 모든 방법에서 2,000 텍스트 토큰이다.
LLM 호출 상한은 설정에서 선택된 질문 × 3방법 × 반복 수다. 개발 각 20·1회이면 120회, 평가 각 50·1회이면 300회다.
품질 평가와 독립 검색 지연 측정에도 C API 호출이 발생한다. 자동 재시도는 없고 오류 전 완료 결과는 `results/partial/`에 남는다.

In [ ]:
system_results = []
system_directory = None
if RUN_GENERATION and engine:
    from openai import OpenAI, DefaultHttpxClient
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('루트 .env에 OPENAI_API_KEY가 필요합니다.')
    system_partial_directory = ROOT/'results/partial'/uuid4().hex
    system_partial_directory.mkdir(parents=True,exist_ok=False)
    meta = runtime_manifest(ROOT,engine)
    meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'live_system_abc','generation':GENERATION,'split':SPLIT,'repeats':REPEATS,
                 'notebook_source_sha256':notebook_source_sha256(ROOT/'notebook/05_measure_abc.ipynb'),
                 'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
    write_json(system_partial_directory/'manifest.json',meta)
    def save_system_record(record):
        with (system_partial_directory/'system.jsonl').open('a',encoding='utf-8') as stream:
            stream.write(json.dumps(record,ensure_ascii=False)+'\n')
    try:
        with DefaultHttpxClient(timeout=180,event_hooks=RAW_LOG.http_hooks if RAW_LOG else {}) as llm_http, \
             OpenAI(max_retries=0,timeout=180,http_client=llm_http) as generator:
            system_results = measure_system(engine,questions,labels,client=generator,
                generation_config=GENERATION,max_calls=MAX_LLM_API_CALLS,repeats=REPEATS,methods=METHODS,
                seed=CONFIG['seed'],on_result=save_system_record,audit=RAW_LOG)
    finally:
        jev_client.close()
        if jev_http: jev_http.close()
    system_directory = save_run(ROOT,[r['retrieval'] for r in system_results],meta,'05_measure_abc.ipynb')
    write_jsonl(system_directory/'system.jsonl',system_results)
    write_json(system_partial_directory/'completed.json',{'run_directory':str(system_directory)})
    print('System saved:',system_directory)
elif jev_client:
    jev_client.close()
    if jev_http: jev_http.close()
    print('생성 미실행. 검색만 측정했습니다.')

In [ ]:
if system_results:
    rows = []
    for r in system_results:
        retrieved,generated = r['retrieval'],r['generation']
        usage = generated['api_usage'] or {}
        rows.append({'question_id':retrieved['question_id'],'task':retrieved['task'],'method':retrieved['method'],
                     'repeat':retrieved['repeat'],'online_total_ms':r['online_total_seconds']*1000,
                     'diagnostic_capture_ms':r.get('diagnostic_capture_seconds',0)*1000,
                     'raw_events_path':(r.get('raw_log') or {}).get('events_path'),
                     'answer':generated['answer'],'valid_output':generated['valid_output'],
                     'evidence_count':len(retrieved['evidence']),
                     'complete_evidence_hit':retrieved['metrics']['complete_evidence_hit'],
                     'jev_input_tokens':(retrieved.get('api_usage') or {}).get('input_tokens'),
                     'llm_input_tokens':usage.get('input_tokens'),'llm_output_tokens':usage.get('output_tokens'),
                     'retrieval_api_cost_usd':r['retrieval_api_cost_usd'],
                     'generation_api_cost_usd':r['generation_api_cost_usd'],
                     'online_api_cost_usd':r['online_api_cost_usd'],**r['answer_metrics']})
    system_table = pd.DataFrame(rows)
    system_table.to_csv(system_directory/'system.csv',index=False)
    display(system_table)
    system_summary = system_table.groupby(['task','method']).agg(
        questions=('question_id','nunique'),answer_em=('answer_em','mean'),answer_f1=('answer_f1','mean'),
        p50_ms=('online_total_ms','median'),api_cost_usd=('online_api_cost_usd',lambda x:x.sum(min_count=1)))
    system_summary.to_csv(system_directory/'system_summary.csv')
    display(system_summary)
    print('Measured split:',SPLIT,'| questions per task:',QUESTIONS_PER_TASK,'| repetitions:',REPEATS)
else:
    print('No measured LLM API results.')
if RAW_LOG:
    RAW_LOG.event('session','notebook_finished',{'quality_directory':str(quality_directory) if quality_directory else None,
        'online_directory':str(online_directory) if online_directory else None,
        'system_directory':str(system_directory) if system_directory else None})
    print('Question/method/phase/repeat logs:',RAW_LOG.directory)


## 해석과 다음 실험

초기 C 정책은 분기 확률 0.45, 근거 Score 1.5, 완결성 확률 0.95, 최대 32 API 호출·128 원문 평가·256 노드·8 재탐색이다. Jev의 확률을 정확도로 간주하지 않는다. 트리 설명 누락, 잘못된 적재, 분기 탈락, 잘못된 완결성 판단을 trace에서 확인한다. 자료 개수는 고정하지 않지만 예산 초과·누락은 그대로 기록한다.

다음 단계는 개발셋에서 C 정책을 조정하고 동결한 뒤, 공통 평가 1,000문항과 2,000/10,000/40,000 규모에서 반복 측정하는 것이다. 큰 corpus는 먼저 04에서 별도 적재한다. 준비 비용·사람의 분류체계 작성 시간·업데이트 비용도 최종 연구에 포함해야 한다.

Jev 계약: [API](https://docs.typesafe.ai/api), [고정 모델](https://docs.typesafe.ai/models), [Noul](https://docs.typesafe.ai/primitives/noul), [Score](https://docs.typesafe.ai/primitives/score). 상세 구현 규칙은 `research/c-method.md`를 참고한다.